In [1]:
import sys
from pathlib import Path
PIPELINE_DIR = Path.cwd().parent
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))
import config
from rx import io, text, places, remote

In [2]:
files_raw = io.read_stage("interim", "files_raw")
pages_index = io.read_stage("interim", "pages_index")
pages_decl = io.read_stage("interim", "pages_decl")
state_mentions = io.read_stage("interim", "state_mentions")

# 2.1
for name, df in {"files_raw": files_raw, "pages_index": pages_index,
                 "pages_decl": pages_decl, "state_mentions": state_mentions}.items():
    print(f"{name:15} {len(df):>9,} rows  {df.NAID.nunique():>6,} files")

# 2.2
print(files_raw.n_pages.describe().round(1))
files_raw.nlargest(3, "n_pages")[["title", "n_pages"]]

# 2.3
src = pages_index[["has_transcription", "has_revx_ocr", "has_fs_text"]]
print((src.mean() * 100).round(1))
print("pages with no text:", (~src.any(axis=1)).sum())
print("files with any transcription: %.1f%%" % (pages_index.groupby("NAID").has_transcription.any().mean() * 100))

# 2.4
print(files_raw.title.str.extract(r"File\s+([A-Z])")[0].value_counts(dropna=False))
files_raw.title[~files_raw.title.str.contains(r"File\s+[A-Z]")].value_counts().head(5)


files_raw          78,745 rows  78,745 files
pages_index     2,195,663 rows  78,745 files
pages_decl        334,409 rows  63,030 files
state_mentions    204,846 rows  65,624 files
count    78745.0
mean        27.9
std         23.7
min          1.0
25%         12.0
50%         22.0
75%         39.0
max        609.0
Name: n_pages, dtype: float64
has_transcription    40.3
has_revx_ocr         99.3
has_fs_text          99.5
dtype: float64
pages with no text: 1441
files with any transcription: 40.0%
0
S      36114
W      25622
R      11164
B       3237
NaN     2501
O         61
C         16
N         15
D          4
F          2
A          2
M          2
P          2
V          1
J          1
T          1
Name: count, dtype: int64


title
Revolutionary War Pension and Bounty Land Warrant Application File                                                      1445
Revolutionary War Pension and Bounty Land Warrant Application Files - NARA Archival Administrative Sheets                303
Revolutionary War Pension and Bounty Land Warrant Application File [Blank], for [Blank] [Blank], [Blank]                  10
Revolutionary War Pension and Bounty Land Warrant Application File [ILLEGIBLE], [ILLEGIBLE] [ILLEGIBLE], [ILLEGIBLE]       8
Revolutionary War Pension and Bounty Land Warrant Application File [ILLEGIBLE], [ILLEGIBLE] [ILLEGIBLE], Penn.             2
Name: count, dtype: int64

In [3]:
pages_decl = io.read_stage("interim", "pages_decl")
reason_pages = io.read_stage("interim", "reason_pages")
files_raw = io.read_stage("interim", "files_raw")

# 2.5
print(pages_decl.groupby("NAID").size().describe().round(1))
print(pages_decl.kind.value_counts(), pages_decl.offs.value_counts().sort_index(), pages_decl.text_source.value_counts())

# 2.6
w_ids = files_raw[files_raw.title.str.contains(r"File\s+W(?![a-z])")].NAID
x = pages_decl[pages_decl.NAID.isin(w_ids)].NAID.sample(1, random_state=4).iloc[0]
for r in pages_decl[pages_decl.NAID == x].sort_values("pageNumber").itertuples():
    print("----", r.pageNumber, r.kind, r.offs, "| widow" if "widow" in str(r.text).lower() else "")
    print(str(r.text)[:400])

# 2.7
for t in reason_pages.text.sample(5, random_state=1):
    i = t.find("reason")
    print("…", t[i:i + 250], "\n")

count    63030.0
mean         5.3
std          3.3
min          1.0
25%          3.0
50%          3.0
75%          6.0
max         66.0
dtype: float64
kind
declaration    316194
fallback        18215
Name: count, dtype: int64 offs
0    116443
1    112880
2    105086
Name: count, dtype: int64 text_source
RevX_OCR_text        202040
transcriptionText    132214
fs_extractedText        114
Name: count, dtype: int64
---- 6 declaration 0 | widow
Declaration
In order to obtain the benefits of the third section of the act of Congress of the 7 th July 1838
State of Connecticut
District & County of Tolland - ss.
On this 9 th day of May 1848 personally appeared before me Elisha Stearns Judge of probate for said District of Tolland Esther Fish a resident of said County & District of Tolland aged 81 years who being first duly sworn according to 
---- 7 declaration 1 
John Ladd of Wallasey and Esther Wood Ladd were married December the eleventh day (in the year of our Lord one thousand eight hundred